# Get Raw Data

- Reads the notes directly from BigQuery where is the MIMIC-IV database and the discharge dataset that has the discharge notes; 
- Pulls out the two admission and discharge medication sections with regex, and writes the result to DATA_DIR.

**Note:** 
- this step does NOT use an LLM. 
- The extraction is plain text-matching and runs entirely on this machine;

## 1. Setup & Imports

In [ ]:
import re

import pandas as pd
from google.cloud import bigquery
from pathlib import Path


from clinical_notes_extraction.config import QUERIES_DIR, DATA_DIR, GCP_BILLING_PROJECT, QUERY_FILE_NAME


## 2. Extract Discharge Notes from MIMIC-IV

In [ ]:
def load_query(filename: str, **params) -> str:
    """Read queries/<filename> and substitute {placeholders}."""
    text = (QUERIES_DIR / filename).read_text(encoding="utf-8")
    return text.format(**params) if params else text



def fetch_data(query_file) -> pd.DataFrame:
    """Read the discharge notes directly from BigQuery into a DataFrame."""
    client = bigquery.Client(project=GCP_BILLING_PROJECT)
    sql = load_query(query_file)
    print(f"Querying BigQuery (billing project: {GCP_BILLING_PROJECT})")
    df = client.query(sql).to_dataframe(progress_bar_type="tqdm")
    print(f"Fetched {len(df)} notes")
    return df


In [ ]:
df = fetch_data(QUERY_FILE_NAME)


In [ ]:
df.head()

In [ ]:
df.info()

## 3. Extract Admission and Discharge Medication Sections from Discharge Notes

- Pulls out the two admission and discharge medication sections with regex, and writes the result to DATA_DIR.

In [ ]:
def extract_section(text, header):
    """Return the body of one section from a single note, or None if absent.

    'text'   = the full note (one string)
    'header' = the section title to find, e.g. "Discharge Medications"
    """
    if not isinstance(text, str):  # some notes may be empty / missing
        return None

    # Find the title, keep everything after it (.*?), and stop at the next
    # section title or the end of the note.
    pattern = (
        re.escape(header) + r"\s*:?\s*\n?"
        r"(.*?)"
        r"(?=\n[ \t]*\n?[A-Z][A-Za-z /]+:\s*\n|\Z)"
    )
    match = re.search(pattern, text, re.DOTALL)  # DOTALL = allow multi-line
    return match.group(1).strip() if match else None


def add_medication_sections(df: pd.DataFrame) -> pd.DataFrame:
    """Add the two medication-section columns to a copy of the DataFrame."""
    out = df.copy()

    out["meds_on_admission"] = out["text"].apply(
        lambda note: extract_section(note, "Medications on Admission")
    )
    out["meds_on_discharge"] = out["text"].apply(
        lambda note: extract_section(note, "Discharge Medications")
    )

    # Coverage: what fraction of notes did we actually find each section in?
    print(f"Admission meds found in {100 * out['meds_on_admission'].notna().mean()} of notes")
    print(f"Discharge meds found in {100 * out['meds_on_discharge'].notna().mean()} of notes")
    return out

In [ ]:
df1 = add_medication_sections(df)

In [ ]:
df1.head()

In [ ]:
df1.info()

In [ ]:
out_path = DATA_DIR / "discharge_notes_with_meds.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
df1.to_parquet(out_path, index=False)  # swap for .to_csv if pyarrow isn't installed
print(f"Wrote {len(df1)} rows to {out_path}" )